In [1]:
#|  echo: false
#| output: false # Suppress warnings in html version

print("import")

import os, sympy, copy, nest_asyncio
nest_asyncio.apply()

from petsc4py import PETSc
import underworld3 as uw
import numpy as np
import scipy as sp
import pyvista as pv
import underworld3.visualisation as vis
from IPython.display import IFrame
import matplotlib.pyplot as plt

import


[Wombat.local:73553] shmem: mmap: an error occurred while determining whether or not /var/folders/bt/jsh174p911lcyp7hslpkzktc0000gn/T//ompi.Wombat.501/jf.0/1359413248/sm_segment.Wombat.501.51070000.0 could be created.


In [2]:
print("functions set up")

def Save_TS_to_list(Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list,
                    swarm, time, Stress, Strainrate, material):
    #### Function to store simulation data in lists. It must be lists because the information is stored on particles which 
    #### flow out, so the number of element at each time step changes
    with swarm.access(material):
        Stress_list.append(    np.hstack([uw.function.evalf(Stress.sym,     swarm.particle_coordinates.data)[:,0,:],
                                          uw.function.evalf(Stress.sym,     swarm.particle_coordinates.data)[:,1,:]]))
        Strainrate_list.append(np.hstack([uw.function.evalf(Strainrate.sym, swarm.particle_coordinates.data)[:,0,:],
                                          uw.function.evalf(Strainrate.sym, swarm.particle_coordinates.data)[:,1,:]]))
        Vel_list.append(uw.function.evalf(Velocity.sym, swarm.particle_coordinates.data))
        Position_list.append(swarm.particle_coordinates.data.copy())
        time_list.append(time)
        Mat_mask_list.append((material.data==1)[:,0])
    return(Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list)

def RepopulateLeftBoundaryAfterAdvenction(initial_left_boundary_particles_positions, swarm):
    #### Function to add particles on the left boudary after dome flow out from the right boundary.
    #### The function takes the intial coordinates of the particles and check which coordinates are too far from all existing particles. 
    #### It than adds those particles back to the swarm. 
    #### Note: it is assumed new particles have the properties of material=0. I didn;t check yet if the new particles need to have 
    ####       their properties defined manually
    particles_to_add = []
    with swarm.access():
        for particle in initial_left_boundary_particles_positions:
            if (min(np.linalg.norm(swarm.data-particle, axis=1)) > 0.1):
                particles_to_add.append(particle)
        ##################################################################################################################################
        #### This part gives an error: 
        #### Cell In[3], line 59, in RepopulateLeftBoundaryAfterAdvenction(initial_left_boundary_particles_positions, swarm)
        #### -> 59     swarm.add_particles_with_coordinates(np.array(particles_to_add))
        #### File /data1/underworld3/src/underworld3/swarm.py:1139, in Swarm.add_particles_with_coordinates(self, coordinatesArray)
        #### -> 1139 self.dm.addNPoints(npoints=npoints)
        #### File petsc4py/PETSc/DMSwarm.pyx:345, in petsc4py.PETSc.DMSwarm.addNPoints()
        #### Error: error code 83
        
        # if np.any(particles_to_add):
        #     swarm.add_particles_with_coordinates(np.array(particles_to_add))        
        ##################################################################################################################################

def TensorSolver(Tensor_Field, Scalar_Field, function, mesh):
    #### Function to set solvers for the Stress and Strainrate fields
    projector             = uw.systems.Tensor_Projection(mesh, tensor_Field=Tensor_Field, scalar_Field=Scalar_Field)
    projector.uw_function = function
    projector.smoothing   = 1.0e-3
    return(projector)

def SetBC(ShearVel, model, FixedShearTest):
    #### Function to set the boundary conditions on all sides:
    ####    Top - prescribed X vel
    ####    Bottom - fixed
    ####    Left+Right - open/only X vel
    ####    Back (for 3D) - free slip (to prevent rotation)
    ####    Front (for 3D) - open (for shear test with some deformation on the second horizontal direction) / 
    ####                     free slip (for fixed shear test with no motion on the second horizontal direction)
    model.essential_bcs = [] # reset BC
    Prescribed_X_vel   = model.mesh.CoordinateSystem.unit_e_0 * ShearVel
    Fixed              = model.mesh.CoordinateSystem.unit_e_0 * 0
    Only_X_vel         = model.mesh.CoordinateSystem.unit_e_0 * sympy.oo
    FreeSlip_FrontBack = sympy.Matrix([sympy.oo, 0, sympy.oo])
    
    model.add_dirichlet_bc(Prescribed_X_vel, "Top")
    model.add_dirichlet_bc(Fixed,            "Bottom")
    # model.add_dirichlet_bc(Only_X_vel,       "Left")
    # model.add_dirichlet_bc(Only_X_vel,       "Right")
    if model.mesh.dim==3:
        model.add_dirichlet_bc(FreeSlip_FrontBack, "Back")
        if FixedShearTest: model.add_dirichlet_bc(FreeSlip_FrontBack, "Front") 


functions set up


In [3]:
print("environment set up")
MeshDim           = 2    #### 2->2D; 3->3D
end_time          = 8.
transition_time   = 1.   #### time for which the sheering stops
materials         = 2
FixedShearTest    = True #### relevant only for 3D
ShearVel          = 1.   #### initial shearing velocity
eta               = 1.   #### shear viscosity
mu                = 1.   #### shear modulus
dt, dt_e          = 0.2, 1.
domain_boundaries = np.array([[-10,0,0],[10,1,1]])
Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list = [], [], [], [], [], []

print("mesh")
mesh = uw.meshing.StructuredQuadBox(elementRes = tuple(5 * np.diff(domain_boundaries, axis=0)[0,:MeshDim]), 
                                    minCoords  = tuple(domain_boundaries[0,:MeshDim]),
                                    maxCoords  = tuple(domain_boundaries[1,:MeshDim]))
Velocity     = uw.discretisation.MeshVariable("V",    mesh, mesh.dim,             vtype=uw.VarType.VECTOR,     degree=2, continuous=True, varsymbol=r"\mathbf{v}")
Pressure     = uw.discretisation.MeshVariable("P",    mesh, 1,                    vtype=uw.VarType.SCALAR,     degree=1, continuous=True, varsymbol=r"p")
Stress       = uw.discretisation.MeshVariable("S",    mesh, (mesh.dim, mesh.dim), vtype=uw.VarType.SYM_TENSOR, degree=2, continuous=True, varsymbol=r"{\sigma}")
Strainrate   = uw.discretisation.MeshVariable("E",    mesh, (mesh.dim, mesh.dim), vtype=uw.VarType.SYM_TENSOR, degree=2, continuous=True, varsymbol=r"{\dot\varepsilon}")
SclrMeshVar1 = uw.discretisation.MeshVariable("SMV1", mesh, 1,                    vtype=uw.VarType.SCALAR,     degree=2, continuous=True, varsymbol=r"SMV1")
SclrMeshVar2 = uw.discretisation.MeshVariable("SMV2", mesh, 1,                    vtype=uw.VarType.SCALAR,     degree=2, continuous=True, varsymbol=r"SMV2")

print("swarm")
swarm            = uw.swarm.Swarm(mesh=mesh)
material         = uw.swarm.IndexSwarmVariable("Material", swarm, indices=materials)
stress_star      = uw.swarm.SwarmVariable(r"stress_dt",    swarm, (mesh.dim, mesh.dim), vtype=uw.VarType.SYM_TENSOR, proxy_continuous=True, proxy_degree=2, varsymbol=r"{\tau^{*}_{p}}")
stress_star_star = uw.swarm.SwarmVariable(r"stress_2dt",   swarm, (mesh.dim, mesh.dim), vtype=uw.VarType.SYM_TENSOR, proxy_continuous=True, proxy_degree=2, varsymbol=r"{\tau^{**}_{p}}")

swarm.populate_petsc(1)
with swarm.access(material), mesh.access():
    material.data[:] = 0
    if materials==2: material.data[(swarm.data[:,0] >= 0) & (swarm.data[:,0] <= 1)] = 1 # 2 materials setup
    #### repopulation
    threshold = swarm.data[:,0].min() + ShearVel*dt # gives the X position the leftmost particle is expected to flow to under the prescribed top BC
    initial_left_boundary_particles_positions = swarm.data[(swarm.data[:,0] < threshold)]

print("stokes set up")
stokes = uw.systems.VE_Stokes(mesh, velocityField=Velocity, pressureField=Pressure, verbose=False, degree=1)
stokes.constitutive_model = uw.constitutive_models.ViscoElasticPlasticFlowModel
stokes.constitutive_model.Parameters.shear_viscosity_0 = material.createMask([eta]*materials)
stokes.constitutive_model.Parameters.shear_modulus     = material.createMask([mu]*materials)
stokes.constitutive_model.Parameters.dt_elastic        = sympy.sympify(dt_e)
stokes.constitutive_model.Parameters.stress_star       = stress_star.sym
stokes.constitutive_model.Parameters.stress_star_star  = stress_star_star.sym

print("stress and strainrate solvers")
stess_projector      = TensorSolver(Stress,     SclrMeshVar1, stokes.stress_deviator, mesh)
strainrate_projector = TensorSolver(Strainrate, SclrMeshVar2, stokes.strainrate,      mesh)

print("bc")
SetBC(ShearVel, stokes, FixedShearTest)

print("initial solve")
time = 0.
Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list = Save_TS_to_list(Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list,
                                                                                                  swarm, time, Stress, Strainrate, material)

             

environment set up
mesh
Structured box element resolution 100 5
swarm
stokes set up
stress and strainrate solvers
bc
initial solve


In [6]:
stokes._build()

In [10]:
stokes.F1.sym[0,0]

({ {\Delta t_{e}} \hspace{ 0.07pt } }*{ {\mu} \hspace{ 0.06pt } }*({ \uplambda \hspace{ 0.0pt } }*({\mathbf{v}}_{ 0,0}(N.x, N.y) + {\mathbf{v}}_{ 1,1}(N.x, N.y)) - {p}(N.x, N.y)) + { {\eta_{\mathrm{eff}}} \hspace{ 0.12pt } }*(4*{ {\Delta t_{e}} \hspace{ 0.07pt } }*{ {\mu} \hspace{ 0.06pt } }*{\mathbf{v}}_{ 0,0}(N.x, N.y) + 4*{{ {F[ {\mathbf{v}} ] }^{ * } }}_{ 00 }(N.x, N.y) - {{ {F[ {\mathbf{v}} ] }^{ ** } }}_{ 00 }(N.x, N.y))/2)/({ {\mu} \hspace{ 0.06pt } }*{ {\Delta t_{e}} \hspace{ 0.07pt } })

In [19]:
stokes.constitutive_model.stress_2star.sym

Matrix([
[{{ {F[ {\mathbf{v}} ] }^{ ** } }}_{ 00 }(N.x, N.y), {{ {F[ {\mathbf{v}} ] }^{ ** } }}_{ 01 }(N.x, N.y)],
[{{ {F[ {\mathbf{v}} ] }^{ ** } }}_{ 01 }(N.x, N.y), {{ {F[ {\mathbf{v}} ] }^{ ** } }}_{ 11 }(N.x, N.y)]])

In [ ]:
0/0

In [ ]:
stokes.solve(zero_init_guess=True, timestep=dt)
strainrate_projector.solve()
stess_projector.solve()
time += dt
Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list = Save_TS_to_list(Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list,
                                                                                                  swarm, time, Stress, Strainrate, material)
print("loop")
while time <= end_time:#10.5:#10.05:
    if time >= transition_time: SetBC(0., stokes, FixedShearTest)
    print("  time {}\n    solve".format(time))
    swarm.advection(stokes.u.sym, dt, order=2, evalf=True) # advect the swarm
    RepopulateLeftBoundaryAfterAdvenction(initial_left_boundary_particles_positions, swarm)
    stokes.solve(zero_init_guess=False, timestep=dt)
    strainrate_projector.solve()
    stess_projector.solve()
    time = np.around(time+dt,5)
    Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list = Save_TS_to_list(Position_list, time_list, Stress_list, Strainrate_list, Mat_mask_list, Vel_list,
                                                                                                      swarm, time, Stress, Strainrate, material)
time_list = np.array(time_list)

print("done")

In [ ]:
#### Visualise last step

print_step = 3 #-1

pvmesh = vis.mesh_to_pv_mesh(stokes.mesh)
pvmesh.point_data["U"] = vis.vector_fn_to_pv_points(pvmesh, stokes.u.sym)
pvmesh.point_data["P"] = vis.scalar_fn_to_pv_points(pvmesh, stokes.p.sym)

points       = np.zeros((len(Stress_list[print_step][:,0]), 3))
points[:, 0] = Position_list[print_step][:, 0]
points[:, 1] = Position_list[print_step][:, 1]
if mesh.dim==3:    points[:, 2] = Position_list[print_step][:, 2]
point_cloud = pv.PolyData(points)
point_cloud.point_data["Stress XY"] = Stress_list[print_step][:,1].copy()

mask = Mat_mask_list[print_step]
points_mask       = np.zeros((len(Stress_list[print_step][mask,0]), 3))
points_mask[:, 0] = Position_list[print_step][mask, 0]
points_mask[:, 1] = Position_list[print_step][mask, 1]
point_cloud_mask  = pv.PolyData(points_mask)

print("min stress (mask) = ",  min(Stress_list[print_step][mask,1]))
print("max stress (mask) = ",  max(Stress_list[print_step][mask,1]))
print("mean stress (mask) = ", np.mean(Stress_list[print_step][mask,1]))
print("min stress (all) = ",   min(Stress_list[print_step][:,1]))
print("max stress (all) = ",   max(Stress_list[print_step][:,1]))
print("mean stress (all) = ",  np.mean(Stress_list[print_step][:,1]))
print("U max = ", pvmesh.point_data["U"].max())
print("U min = ", pvmesh.point_data["U"].min())

V_Mag     = 0.5/np.abs(pvmesh.point_data["U"]).max()
sig_mean  = np.mean(Stress_list[print_step][mask,1])
sig_delta = max(Stress_list[print_step][mask,1]) - min(Stress_list[print_step][mask,1])

pl = pv.Plotter(window_size=(1250, 750))
pl.add_mesh(pvmesh, color='white', edge_color="gray", show_edges=True)
pl.add_points(point_cloud,      render_points_as_spheres=False, point_size=5, opacity=0.25, show_scalar_bar=True,  
              scalars="Stress XY", cmap="seismic", clim=[sig_mean-sig_delta, sig_mean+sig_delta])
pl.add_points(point_cloud_mask, render_points_as_spheres=False, point_size=5, opacity=0.5,  show_scalar_bar=False, 
              color='yellow')
pl.add_arrows(pvmesh.points, pvmesh.point_data["U"], mag=V_Mag, color='black')
pl.show()

In [ ]:
#### Plot stress and strainrate over time
#### Note: plotting data for the yellow section to reduce boundary effects

### Analytical solution calculation
def Calc_Ref(t1):
    h = np.diff(domain_boundaries, axis=0)[0,1]
    C1 = -(ShearVel**2)*(eta**2)*mu/((mu**2)*(h**2)+(ShearVel**2)*(eta**2))
    C2 = -ShearVel*h*eta*(mu**2)/((mu**2)*(h**2)+(ShearVel**2)*(eta**2))

    if t1<transition_time:
        tau = (np.exp(-(mu/eta)*t1)              * (C2*np.cos(ShearVel*t1/h)              - C1*np.sin(ShearVel*t1/h))              - C2)
    elif t1>=transition_time:
        tau = (np.exp(-(mu/eta)*transition_time) * (C2*np.cos(ShearVel*transition_time/h) - C1*np.sin(ShearVel*transition_time/h)) - C2)  * np.exp(-(mu/eta)*(t1-transition_time))
    return(tau)

new_time_list = []
min_sig_XX,  max_sig_XX,  min_sig_XY,  max_sig_XY,  min_sig_YX,  max_sig_YX,  min_sig_YY,  max_sig_YY  = [],[],[],[],[],[],[],[]
min_edot_XX, max_edot_XX, min_edot_XY, max_edot_XY, min_edot_YX, max_edot_YX, min_edot_YY, max_edot_YY = [],[],[],[],[],[],[],[]
ref_sig_XY,  tau_stor_XY = [],[]

for idx, Time in enumerate(time_list):
    ref_sig_XY.append(Calc_Ref(Time))

    mask = Mat_mask_list[idx]
    new_time_list.append(time_list[idx])
    max_sig_XX.append(Stress_list[idx][mask,0].max())
    min_sig_XX.append(Stress_list[idx][mask,0].min())
    max_sig_XY.append(Stress_list[idx][mask,1].max())
    min_sig_XY.append(Stress_list[idx][mask,1].min())
    max_sig_YX.append(Stress_list[idx][mask,2].max())
    min_sig_YX.append(Stress_list[idx][mask,2].min())
    max_sig_YY.append(Stress_list[idx][mask,3].max())
    min_sig_YY.append(Stress_list[idx][mask,3].min())

    max_edot_XX.append(Strainrate_list[idx][mask,0].max())
    min_edot_XX.append(Strainrate_list[idx][mask,0].min())
    max_edot_XY.append(Strainrate_list[idx][mask,1].max())
    min_edot_XY.append(Strainrate_list[idx][mask,1].min())
    max_edot_YX.append(Strainrate_list[idx][mask,2].max())
    min_edot_YX.append(Strainrate_list[idx][mask,2].min())
    max_edot_YY.append(Strainrate_list[idx][mask,3].max())
    min_edot_YY.append(Strainrate_list[idx][mask,3].min())

fig, ax = plt.subplots(nrows=2, ncols=1)
ax[0].plot(new_time_list, max_sig_XY, marker='.', label="max sig_XY")
ax[0].plot(new_time_list, ref_sig_XY, marker='.', label="ref sig_XY")
ax[0].legend()

ax[1].plot(new_time_list, max_edot_YX, marker='.', label="max edot_YX")
ax[1].legend()
plt.show()